In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS catalog_smartfactory.gold.checkpoints")

DataFrame[]

In [0]:
silver_stream_df = (
    spark.readStream
    .table("catalog_smartfactory.silver.streaming_iot_telemetry")
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

machine_health_kpi_query = (
    silver_stream_df
    .groupBy("machine_id","event_date")
    .agg(
        sum("temperature_alerts").alias("total_temp_alerts"),
        sum("pressure_alerts").alias("total_pressure_alerts"),
        sum("vibration_alerts").alias("total_vibration_alerts"),
        round(avg("health_score"),2).alias("avg_health_score"),
        round(avg("failure_risk_score"),2).alias("avg_failure_risk_score"),

    )
)

In [0]:
daily_operations_kpi_query  = (
    silver_stream_df
    .groupBy("event_date")
    .agg(
        countDistinct("machine_id").alias("active_machines"),
        round(avg("temperature"),2).alias("avg_temperature"),
        round(avg("vibration"),2).alias("avg_vibration"),
        round(avg("pressure"),2).alias("avg_pressure"),
        round(avg("health_score"),2).alias("avg_health_score"),
        sum("temperature_alerts").alias("temperatur_trend"),
        sum("pressure_alerts").alias("pressure_trend"),
        sum("vibration_alerts").alias("vibration_trend"),
    )
)

In [0]:
failure_risk_kpi_query  = (
    silver_stream_df
    .filter(col("failure_risk_score") > 0.6)
    .select(
        "machine_id",
        "timestamp",
        "temperature",
        "vibration",
        "pressure",
        "failure_risk_score",
        "health_score",
        "machine_status"
    )
)

In [0]:
machine_health_kpi = (
    machine_health_kpi_query.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option(
            "checkpointLocation",
            "/Volumes/catalog_smartfactory/gold/checkpoints/iot_gold_streaming_v1"
    )
    .outputMode("Complete")
    .toTable("catalog_smartfactory.gold.machine_health_kpi")
)

In [0]:
machine_health_kpi.stop()

In [0]:
daily_operations_kpi = (
    machine_health_kpi_query.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option(
            "checkpointLocation",
            "/Volumes/catalog_smartfactory/gold/checkpoints/iot_gold_streaming_daily_v1"
    )
    .outputMode("Complete")
    .toTable("catalog_smartfactory.gold.daily_operations_kpi")
)

In [0]:
daily_operations_kpi.stop()

In [0]:
failure_risk_kpi = (
    machine_health_kpi_query.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option(
            "checkpointLocation",
            "/Volumes/catalog_smartfactory/gold/checkpoints/iot_gold_streaming_risk_v2"
    )
    .outputMode("Complete")
    .toTable("catalog_smartfactory.gold.failure_risk_kpi")
)

In [0]:
failure_risk_kpi.stop()